In [ ]:
# 0. Cai dung phien ban thu vien
%pip install -q ultralytics==8.4.183 aiohttp


In [ ]:
# 1. GPU, Drive va NDJSON goc
from pathlib import Path
from google.colab import drive
import json, hashlib, shutil, torch, ultralytics
assert torch.cuda.is_available(), 'Can ket noi GPU truoc khi train.'
print('GPU:', torch.cuda.get_device_name(0), '| Ultralytics:', ultralytics.__version__)
MOUNT_DIR=Path('/CV_Object_Detection/drive')
MOUNT_DIR.parent.mkdir(parents=True,exist_ok=True)
drive.mount(str(MOUNT_DIR))
DRIVE_DIR=MOUNT_DIR/'MyDrive/YOLO_Grocerystuffs'
WORK_DIR=Path('/content/grocerystuffs')
SNAPSHOT_DIR=DRIVE_DIR/'datasets/grocerystuffs_20261029'
for folder in (DRIVE_DIR,WORK_DIR,SNAPSHOT_DIR): folder.mkdir(parents=True,exist_ok=True)
SOURCE_SHA='65dc7d0a947a98ff0adfbc8654a2d2c0c459ab06d7d6b8f6769b8c147988dac3'
SOURCE_PATH=SNAPSHOT_DIR/'source_export.ndjson'
UPLOAD=Path('/content/grocerystuffs.ndjson')
if not SOURCE_PATH.is_file():
    assert UPLOAD.is_file(), 'Upload grocerystuffs.ndjson vao Files cua Colab.'
    assert hashlib.sha256(UPLOAD.read_bytes()).hexdigest()==SOURCE_SHA
    shutil.copy2(UPLOAD,SOURCE_PATH)
assert hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()==SOURCE_SHA
records=[json.loads(s) for s in SOURCE_PATH.read_text(encoding='utf-8-sig').splitlines() if s.strip()]
metadata=next(r for r in records if r.get('type')=='dataset')
image_records=[r for r in records if r.get('type')=='image']
NAMES={int(k):v for k,v in metadata['class_names'].items()}
assert metadata['task']=='segment' and len(image_records)==918 and len(NAMES)==31
print('Drive:',DRIVE_DIR,'| Anh:',len(image_records),'| Lop:',len(NAMES))


In [ ]:
# 2. Chia theo nhom nguon, chuyen NDJSON thanh YOLO segmentation
from collections import Counter,defaultdict
import re, numpy as np, zipfile, yaml
from ultralytics.data.converter import convert_ndjson_to_yolo

def source_group(record):
    stem=Path(record['file']).stem.lower()
    if '.rf.' in stem: return 'rf:'+stem.split('.rf.')[0]
    match=re.match(r'win_(\d{8})_(\d{2})_(\d{2})_',stem)
    if match: return 'camera:'+''.join(match.groups())
    match=re.match(r'img_(\d{8})_(\d{2})(\d{2})',stem)
    if match: return 'camera:'+''.join(match.groups())
    return 'file:'+stem

image_records=[r for r in records if r.get('type')=='image']
groups=defaultdict(list)
for index,record in enumerate(image_records): groups[source_group(record)].append(index)
keys=sorted(groups)
sizes=np.array([len(groups[k]) for k in keys])
class_counts=np.zeros((len(keys),len(NAMES)),dtype=int)
for j,key in enumerate(keys):
    for index in groups[key]:
        for segment in image_records[index].get('annotations',{}).get('segments',[]):
            assert len(segment)>=7 and (len(segment)-1)%2==0
            assert int(segment[0]) in NAMES and all(0<=v<=1 for v in segment[1:])
            class_counts[j,int(segment[0])]+=1
active=class_counts.sum(axis=0)>0
eligible_eval=(class_counts>0).sum(axis=0)>=3
rng=np.random.default_rng(42)
best_score=float('inf'); assignment=None
target=np.array([0.8,0.1,0.1])
for attempt in range(2000):
    candidate=rng.choice(3,size=len(keys),p=target)
    for cls in np.flatnonzero(active):
        if class_counts[candidate==0,cls].sum()==0:
            possible=np.flatnonzero(class_counts[:,cls]>0)
            candidate[rng.choice(possible)]=0
    totals=np.bincount(candidate,weights=sizes,minlength=3)
    if np.any(totals==0): continue
    split_classes=np.array([class_counts[candidate==s].sum(axis=0) for s in range(3)])
    missing_eval=int(((split_classes[1:]==0)&eligible_eval).sum())
    score=1000*float(((totals/len(image_records)-target)**2).sum())+2*missing_eval
    if score<best_score: best_score=score; assignment=candidate.copy()
assert assignment is not None
split_names=('train','val','test')
new_split={index:split_names[int(assignment[j])] for j,key in enumerate(keys) for index in groups[key]}
prepared=[]; image_index=0
for record in records:
    copy=dict(record)
    if copy.get('type')=='image':
        copy['split']=new_split[image_index]; image_index+=1
    prepared.append(copy)
excluded_records=[r for r in prepared if r.get('type')=='image' and not r.get('annotations',{}).get('segments',[])]
prepared=[r for r in prepared if r.get('type')!='image' or r.get('annotations',{}).get('segments',[])]
assert len(excluded_records)==14
TRAIN_SNAPSHOT_DIR=SNAPSHOT_DIR/'labeled_only'
TRAIN_SNAPSHOT_DIR.mkdir(parents=True,exist_ok=True)
prepared_bytes=('\n'.join(json.dumps(r,ensure_ascii=False,separators=(',',':')) for r in prepared)+'\n').encode('utf-8')
PREPARED_SHA=hashlib.sha256(prepared_bytes).hexdigest()
NDJSON_PATH=TRAIN_SNAPSHOT_DIR/'grouped.ndjson'
if NDJSON_PATH.is_file(): assert NDJSON_PATH.read_bytes()==prepared_bytes, 'Grouped snapshot changed; inspect before reuse.'
else: NDJSON_PATH.write_bytes(prepared_bytes)
image_records=[r for r in prepared if r.get('type')=='image']
expected_splits=Counter(r['split'] for r in image_records)
expected_segments=sum(len(r.get('annotations',{}).get('segments',[])) for r in image_records)
expected_empty=sum(not r.get('annotations',{}).get('segments',[]) for r in image_records)
assert len(image_records)==904 and expected_segments==1827 and expected_empty==0
group_splits=defaultdict(set)
for record in image_records: group_splits[source_group(record)].add(record['split'])
assert all(len(splits)==1 for splits in group_splits.values())
train_classes={int(s[0]) for r in image_records if r['split']=='train' for s in r.get('annotations',{}).get('segments',[])}
assert train_classes==set(np.flatnonzero(active).tolist())
print('GROUPED_SPLITS:',dict(expected_splits),'| polygons:',expected_segments,'| empty:',expected_empty)
print('Absent class:',[NAMES[i] for i in NAMES if not active[i]])
print('Prepared SHA256:',PREPARED_SHA)
DATASET_ZIP=TRAIN_SNAPSHOT_DIR/'dataset.zip'
if DATASET_ZIP.is_file():
    DATASET_ROOT=WORK_DIR/'restored'
    DATASET_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as archive:
        for name in archive.namelist(): assert (DATASET_ROOT/name).resolve().is_relative_to(DATASET_ROOT.resolve())
        assert archive.testzip() is None
        archive.extractall(DATASET_ROOT)
    DATA_YAML=DATASET_ROOT/'data.yaml'
else:
    DATA_YAML=Path(await convert_ndjson_to_yolo(str(NDJSON_PATH),output_path=str(WORK_DIR/'converted')))
    DATASET_ROOT=DATA_YAML.parent
assert DATA_YAML.is_file()
config=yaml.safe_load(DATA_YAML.read_text())
config['path']=str(DATASET_ROOT)
config['task']='segment'
for split in split_names: config[split]=f'images/{split}'
DATA_YAML.write_text(yaml.safe_dump(config,allow_unicode=True,sort_keys=False))
print('DATA_YAML:',DATA_YAML)


In [ ]:
# 3. Kiem tra du anh/polygon va luu dataset.zip de resume
from PIL import Image,ImageDraw,ImageOps
from IPython.display import display
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset
from ultralytics.cfg import get_cfg
from urllib.parse import urlsplit
import os, gc
VALIDATED_SHA=None
info=check_det_dataset(str(DATA_YAML))
assert info['names']==NAMES
split_images={}; audit_splits={}; total_segments=0; total_empty=0
for split in ('train','val','test'):
    ds=YOLODataset(img_path=info[split],data=info,task='segment',imgsz=640,augment=False,cache=False,hyp=get_cfg())
    paths=[Path(p) for p in ds.im_files]
    assert len(paths)==expected_splits[split], f'Missing/corrupt image in {split}'
    for p in paths:
        with Image.open(p) as image: image.verify()
    instances=sum(len(label['cls']) for label in ds.labels)
    polygons=sum(len(label['segments']) for label in ds.labels)
    assert instances==polygons
    empty=sum(len(label['cls'])==0 for label in ds.labels)
    audit_splits[split]=dict(images=len(paths),segments=polygons,empty=empty)
    split_images[split]=paths
    total_segments+=polygons; total_empty+=empty
    print(split,audit_splits[split])
    del ds
expected_unique_segments=sum(len({tuple(s) for s in r.get('annotations',{}).get('segments',[])}) for r in image_records)
assert total_segments==expected_unique_segments and total_empty==expected_empty
print('Exact duplicate polygons normalized:',expected_segments-expected_unique_segments)
# File anh cung bytes khong duoc nam o hai split khac nhau.
content_splits=defaultdict(set)
for split,paths in split_images.items():
    for p in paths: content_splits[hashlib.sha256(p.read_bytes()).hexdigest()].add(split)
assert all(len(v)==1 for v in content_splits.values()), 'Exact duplicate images cross splits; resolve before training.'
# Xem polygon tren 8 anh co nhan; 14 anh thieu nhan duoc giu trong source_export.ndjson.
samples=[]
for cls in NAMES:
    match=next((r for r in image_records if any(int(s[0])==cls for s in r.get('annotations',{}).get('segments',[])) and r not in samples),None)
    if match is not None: samples.append(match)
    if len(samples)==8: break
canvas=Image.new('RGB',(4*280,2*230),'#202020'); draw=ImageDraw.Draw(canvas)
all_paths=[p for paths in split_images.values() for p in paths]
for i,r in enumerate(samples):
    stem=Path(urlsplit(r['url']).path).stem
    matches=[p for p in all_paths if p.stem.startswith(stem+'_')]
    assert len(matches)==1
    with Image.open(matches[0]) as im: image=im.convert('RGB')
    overlay=ImageDraw.Draw(image)
    for seg in r.get('annotations',{}).get('segments',[]):
        pts=[(x*image.width,y*image.height) for x,y in zip(seg[1::2],seg[2::2])]
        overlay.line(pts+[pts[0]],fill='lime',width=4)
    thumb=ImageOps.contain(image,(275,200))
    x=(i%4)*280; y=(i//4)*230
    canvas.paste(thumb,(x,y)); draw.text((x+3,y+203),f"{r['split']} | {r['file'][:26]}",fill='white')
print('POLYGON REVIEW - 14 unannotated product photos excluded, originals retained.')
display(canvas)
audit=dict(source_sha256=SOURCE_SHA,prepared_sha256=PREPARED_SHA,task='segment',names=NAMES,
    grouping='Roboflow original stem; camera photos grouped by capture minute; seed42; all observed classes in train',
    excluded_unannotated=[r['file'] for r in excluded_records],raw_segments=expected_segments,unique_segments=expected_unique_segments,splits=audit_splits,missing_classes=[NAMES[i] for i in NAMES if not active[i]])
(TRAIN_SNAPSHOT_DIR/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2))
if not DATASET_ZIP.is_file():
    local_zip=Path(shutil.make_archive(str(WORK_DIR/'dataset_backup'),'zip',root_dir=DATASET_ROOT))
    with zipfile.ZipFile(local_zip) as archive: assert archive.testzip() is None
    temporary_zip=DATASET_ZIP.with_suffix('.zip.partial')
    shutil.copy2(local_zip,temporary_zip)
    assert hashlib.sha256(temporary_zip.read_bytes()).hexdigest()==hashlib.sha256(local_zip.read_bytes()).hexdigest()
    os.replace(temporary_zip,DATASET_ZIP)
VALIDATED_SHA=PREPARED_SHA
gc.collect()
print('VALIDATED:',VALIDATED_SHA,'| backup:',DATASET_ZIP)


In [ ]:
# 4. Train / resume YOLO26l-seg tren Grocerystuffs
from ultralytics import YOLO
import json,hashlib,torch
assert torch.cuda.is_available() and globals().get('VALIDATED_SHA')==PREPARED_SHA
RUN_NAME='Grocerystuffs_Seg_20261029'
RUN_DIR=DRIVE_DIR/'runs'/RUN_NAME
BEST_PT=RUN_DIR/'weights/best.pt'
LAST_PT=RUN_DIR/'weights/last.pt'
PROVENANCE=RUN_DIR/'dataset_provenance.json'
is_resume=LAST_PT.is_file()
if is_resume:
    saved=json.loads(PROVENANCE.read_text())
    assert saved['source_sha256']==SOURCE_SHA and saved['prepared_sha256']==PREPARED_SHA and saved['task']=='segment'
    checkpoint=torch.load(LAST_PT,map_location='cpu',weights_only=False)
    assert checkpoint.get('epoch',-1)>=0 and checkpoint.get('optimizer') is not None, 'Training already finished; do not restart it.'
    print('RESUME from epoch:',checkpoint['epoch']+2)
    del checkpoint
    model=YOLO(str(LAST_PT))
else:
    assert not RUN_DIR.exists(), f'Existing folder without last.pt: {RUN_DIR}; inspect before reuse.'
    model=YOLO('yolo26l-seg.pt')
assert model.task=='segment'
def verify_training(trainer):
    assert trainer.args.task=='segment' and Path(trainer.args.data).resolve()==DATA_YAML.resolve()
    assert trainer.data['names']==NAMES and Path(trainer.save_dir).resolve()==RUN_DIR.resolve()
    if not PROVENANCE.is_file():
        PROVENANCE.write_text(json.dumps(dict(source_sha256=SOURCE_SHA,prepared_sha256=PREPARED_SHA,
            task='segment',pretrained='yolo26l-seg.pt',names=NAMES,splits=audit_splits,
            epochs=100,patience=20,imgsz=640,batch=trainer.batch_size,lr0=0.0005),ensure_ascii=False,indent=2))
    print('VERIFIED_GROCERIES_TRAIN:',trainer.start_epoch+1,'/100 | batch:',trainer.batch_size,'| checkpoint:',trainer.last,flush=True)
def mark_epoch(trainer): trainer._grocery_epoch=trainer.epoch
def save_progress(trainer):
    if trainer.epoch!=getattr(trainer,'_grocery_epoch',None): return
    assert Path(trainer.last).is_file(), 'Checkpoint has not been saved to Drive.'
    if trainer.epoch==trainer.start_epoch:
        print('CHECKPOINT_SAVED:',trainer.last,'| bytes:',Path(trainer.last).stat().st_size,flush=True)
    (RUN_DIR/'progress.json').write_text(json.dumps(dict(epoch=trainer.epoch+1,batch=trainer.batch_size,
        metrics={k:float(v) for k,v in trainer.metrics.items()},last=str(trainer.last)),indent=2))
model.add_callback('on_train_start',verify_training)
model.add_callback('on_train_epoch_end',mark_epoch)
model.add_callback('on_fit_epoch_end',save_progress)
if is_resume:
    results=model.train(resume=True,data=str(DATA_YAML),device=0,workers=2,save_dir=str(RUN_DIR))
else:
    results=model.train(data=str(DATA_YAML),epochs=100,patience=20,imgsz=640,
        batch=0.7,device=0,workers=2,cache=False,amp=True,freeze=None,nbs=64,
        optimizer='AdamW',lr0=0.0005,lrf=0.1,cos_lr=True,weight_decay=0.0005,warmup_epochs=5.0,
        hsv_h=0.015,hsv_s=0.4,hsv_v=0.3,degrees=3.0,translate=0.1,scale=0.3,
        fliplr=0.5,flipud=0.0,mosaic=0.3,close_mosaic=10,mixup=0.0,
        seed=42,deterministic=True,val=True,plots=True,save=True,save_period=5,
        project=str(DRIVE_DIR/'runs'),name=RUN_NAME,exist_ok=False)
RUN_DIR=Path(model.trainer.save_dir)
BEST_PT=RUN_DIR/'weights/best.pt'; LAST_PT=RUN_DIR/'weights/last.pt'
print('TRAIN_FINISHED:',RUN_DIR,'| Best:',BEST_PT,'| Last:',LAST_PT)


In [ ]:
# 5. Xem bieu do sau khi train xong
from IPython.display import Image as DisplayImage,display
assert BEST_PT.is_file()
for name in ('results.png','BoxPR_curve.png','MaskPR_curve.png','confusion_matrix_normalized.png','val_batch0_labels.jpg','val_batch0_pred.jpg'):
    path=RUN_DIR/name
    if path.is_file():
        print(name); display(DisplayImage(filename=str(path),width=1000))


In [ ]:
# 6. Danh gia TEST sau khi chot model - ca bounding box va mask
from ultralytics import YOLO
assert BEST_PT.is_file() and DATA_YAML.is_file()
best_model=YOLO(str(BEST_PT))
test_metrics=best_model.val(data=str(DATA_YAML),split='test',imgsz=640,batch=4,device=0,workers=2,
    plots=True,project=str(RUN_DIR),name='test_evaluation',exist_ok=True)
test_summary=dict(dataset_sha256=PREPARED_SHA,checkpoint_sha256=hashlib.sha256(BEST_PT.read_bytes()).hexdigest(),
    box=dict(precision=float(test_metrics.box.mp),recall=float(test_metrics.box.mr),map50=float(test_metrics.box.map50),map5095=float(test_metrics.box.map)),
    mask=dict(precision=float(test_metrics.seg.mp),recall=float(test_metrics.seg.mr),map50=float(test_metrics.seg.map50),map5095=float(test_metrics.seg.map)))
(RUN_DIR/'test_summary.json').write_text(json.dumps(test_summary,indent=2))
print(json.dumps(test_summary,indent=2))
print('MODEL MOI:',BEST_PT)
